In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/sc_train.bin
/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/acts_train.bin
/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/acts_val.bin
/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/tribunals_val_f.bin
/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/tokenizer.json
/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/tribunals_train_f.bin
/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/test.json
/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/sc_val.bin
/kaggle/input/datasets/achiver7/project-phase-1/data/sc_val.jsonl
/kaggle/input/datasets/achiver7/project-phase-1/data/acts_val.jsonl
/kaggle/input/datasets/achiver7/project-phase-1/data/acts_train.jsonl
/kaggle/input/datasets/achiver7/project-phase-1/data/tribunals_val.jsonl
/kaggle/input/datasets/achiver7/project-phase-1/data/test.json
/kaggle/input/dataset

In [2]:
# ============================================================
# ARCHITECTURE 2 — ONE-CELL FRESH + EFFICIENT PRE-LN RUN
#
# Uses only:
#   /kaggle/input/.../tokenizer.json
#   /kaggle/input/.../*.bin
#
# NEVER loads the old run_arch2_encdec checkpoint.
#
# Model:
#   129,957,888 parameters
#   d_model=768
#   heads=8
#   d_ff=3072
#   6 encoder + 6 decoder layers
#   Pre-LN
#   GELU
#   tied embeddings/output head
#   sinusoidal positions
#
# Efficiency:
#   fused SDPA attention
#   micro-batch=2
#   grad accumulation=8
#   fused AdamW
#   AMP fp16
#   TF32 matmul
#
# Fresh start:
#   old checkpoint directory is NOT read
#   new run directory is deleted/recreated
# ============================================================

import os
import time
import math
import json
import shutil
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from tokenizers import Tokenizer


# ============================================================
# 0. DEVICE / PERFORMANCE
# ============================================================

assert torch.cuda.is_available(), "CUDA GPU required"

DEV = torch.device("cuda")

print("CUDA:", torch.cuda.get_device_name(0))

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

try:
    torch.backends.cuda.matmul.allow_tf32 = True
except Exception:
    pass

try:
    torch.backends.cudnn.allow_tf32 = True
except Exception:
    pass


# ============================================================
# 1. PATHS / CONFIG
# ============================================================

TOKD = (
    "/kaggle/input/datasets/achiver7/"
    "phase1-tokens-v40k-clean1/tokens"
)

# NEW directory: old collapsed checkpoint is never reused.
RUN = "/kaggle/working/run_arch2_preln_efficient"

if os.path.exists(RUN):
    shutil.rmtree(RUN)

os.makedirs(RUN, exist_ok=True)

V = 40000

CFG = dict(
    BLOCK=1024,

    SRC_LEN=768,
    TGT_LEN=256,

    MICRO_BATCH=2,
    GRAD_ACCUM=8,

    LR=3e-4,
    MIN_LR_FRAC=0.1,
    WD=0.1,
    WARMUP=500,

    STEPS=10000,
    CLIP=1.0,

    EVAL_EVERY=500,

    SEED=42,

    MIX=dict(
        acts=0.273,
        sc=0.469,
        tri=0.258,
    ),
)

BLOCK = CFG["BLOCK"]
S = CFG["SRC_LEN"]
TT = CFG["TGT_LEN"]
MB = CFG["MICRO_BATCH"]
GA = CFG["GRAD_ACCUM"]

assert S + TT == BLOCK
assert 50_000_000 <= 129_957_888 <= 300_000_000
assert abs(sum(CFG["MIX"].values()) - 1.0) < 1e-6

print(
    f"Config: SRC={S} "
    f"TGT={TT} "
    f"MB={MB} "
    f"GA={GA} "
    f"effective={MB*GA}"
)

print("RUN:", RUN)
print("FRESH RUN: no previous checkpoint will be loaded.")


# ============================================================
# 2. TOKENIZER
# ============================================================

tokenizer_path = f"{TOKD}/tokenizer.json"

assert os.path.exists(tokenizer_path), tokenizer_path

tok = Tokenizer.from_file(tokenizer_path)

BOS = tok.token_to_id("<bos>")

assert BOS is not None, "Tokenizer has no <bos>"

print("BOS token:", BOS)


# ============================================================
# 3. SEEDS
# ============================================================

np.random.seed(CFG["SEED"])
torch.manual_seed(CFG["SEED"])
torch.cuda.manual_seed_all(CFG["SEED"])


# ============================================================
# 4. EFFICIENT ATTENTION
# ============================================================

class MultiHeadAttention(nn.Module):

    def __init__(
        self,
        d_model,
        n_heads,
        dropout=0.1
    ):
        super().__init__()

        assert d_model % n_heads == 0

        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads

        self.q_proj = nn.Linear(
            d_model,
            d_model
        )

        self.k_proj = nn.Linear(
            d_model,
            d_model
        )

        self.v_proj = nn.Linear(
            d_model,
            d_model
        )

        self.out_proj = nn.Linear(
            d_model,
            d_model
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(
        self,
        query,
        key_value,
        mask=None,
        is_causal=False
    ):

        B, Tq, _ = query.shape
        _, Tk, _ = key_value.shape

        Q = self.q_proj(query)
        K = self.k_proj(key_value)
        V = self.v_proj(key_value)

        Q = Q.view(
            B,
            Tq,
            self.n_heads,
            self.d_head
        ).transpose(1, 2)

        K = K.view(
            B,
            Tk,
            self.n_heads,
            self.d_head
        ).transpose(1, 2)

        V = V.view(
            B,
            Tk,
            self.n_heads,
            self.d_head
        ).transpose(1, 2)

        dropout_p = (
            self.dropout.p
            if self.training
            else 0.0
        )

        if is_causal:

            # Native causal kernel.
            context = F.scaled_dot_product_attention(
                Q,
                K,
                V,
                attn_mask=None,
                dropout_p=dropout_p,
                is_causal=True,
            )

        else:

            if mask is not None:
                mask = mask.to(
                    device=Q.device,
                    dtype=Q.dtype
                )

            context = F.scaled_dot_product_attention(
                Q,
                K,
                V,
                attn_mask=mask,
                dropout_p=dropout_p,
                is_causal=False,
            )

        context = (
            context
            .transpose(1, 2)
            .contiguous()
            .view(
                B,
                Tq,
                self.d_model
            )
        )

        return self.out_proj(context)


# ============================================================
# 5. SINUSOIDAL POSITIONAL ENCODING
# ============================================================

def make_sinusoidal_table(
    max_len,
    d_model
):

    position = torch.arange(
        max_len,
        dtype=torch.float32
    ).unsqueeze(1)

    k = torch.arange(
        0,
        d_model,
        2,
        dtype=torch.float32
    )

    omega = 1.0 / (
        10000 ** (k / d_model)
    )

    table = torch.zeros(
        max_len,
        d_model,
        dtype=torch.float32
    )

    table[:, 0::2] = torch.sin(
        position * omega
    )

    table[:, 1::2] = torch.cos(
        position * omega
    )

    return table


# ============================================================
# 6. FEED-FORWARD
# ============================================================

class FeedForward(nn.Module):

    def __init__(
        self,
        d_model,
        d_ff,
        dropout=0.1
    ):
        super().__init__()

        self.linear1 = nn.Linear(
            d_model,
            d_ff
        )

        self.activation = nn.GELU()

        self.linear2 = nn.Linear(
            d_ff,
            d_model
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(self, x):

        x = self.linear1(x)
        x = self.activation(x)
        x = self.linear2(x)
        x = self.dropout(x)

        return x


# ============================================================
# 7. PRE-LN ENCODER LAYER
# ============================================================

class TransformerEncoderLayerPreLN(
    nn.Module
):

    def __init__(
        self,
        d_model,
        n_heads,
        d_ff,
        dropout=0.1
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(
            d_model
        )

        self.self_attn = MultiHeadAttention(
            d_model=d_model,
            n_heads=n_heads,
            dropout=dropout
        )

        self.norm2 = nn.LayerNorm(
            d_model
        )

        self.ffn = FeedForward(
            d_model=d_model,
            d_ff=d_ff,
            dropout=dropout
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(self, x):

        # Pre-LN:
        # x + Attention(LN(x))

        nx = self.norm1(x)

        x = x + self.dropout(
            self.self_attn(
                nx,
                nx
            )
        )

        # Pre-LN:
        # x + FFN(LN(x))

        x = x + self.ffn(
            self.norm2(x)
        )

        return x


# ============================================================
# 8. PRE-LN ENCODER
# ============================================================

class TransformerEncoderPreLN(
    nn.Module
):

    def __init__(
        self,
        d_model=768,
        n_heads=8,
        d_ff=3072,
        n_layers=6,
        dropout=0.1
    ):
        super().__init__()

        self.layers = nn.ModuleList([
            TransformerEncoderLayerPreLN(
                d_model=d_model,
                n_heads=n_heads,
                d_ff=d_ff,
                dropout=dropout
            )
            for _ in range(n_layers)
        ])

    def forward(self, x):

        for layer in self.layers:
            x = layer(x)

        return x


# ============================================================
# 9. PRE-LN DECODER LAYER
# ============================================================

class TransformerDecoderLayerPreLN(
    nn.Module
):

    def __init__(
        self,
        d_model,
        n_heads,
        d_ff,
        dropout=0.1
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(
            d_model
        )

        self.self_attn = MultiHeadAttention(
            d_model=d_model,
            n_heads=n_heads,
            dropout=dropout
        )

        self.norm2 = nn.LayerNorm(
            d_model
        )

        self.cross_attn = MultiHeadAttention(
            d_model=d_model,
            n_heads=n_heads,
            dropout=dropout
        )

        self.norm3 = nn.LayerNorm(
            d_model
        )

        self.ffn = FeedForward(
            d_model=d_model,
            d_ff=d_ff,
            dropout=dropout
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(
        self,
        x,
        memory
    ):

        # ----------------------------------------------------
        # 1. Causal self-attention
        # ----------------------------------------------------

        nx = self.norm1(x)

        x = x + self.dropout(
            self.self_attn(
                nx,
                nx,
                is_causal=True
            )
        )

        # ----------------------------------------------------
        # 2. Cross-attention
        # Q = decoder
        # K/V = encoder
        # ----------------------------------------------------

        x = x + self.dropout(
            self.cross_attn(
                self.norm2(x),
                memory
            )
        )

        # ----------------------------------------------------
        # 3. FFN
        # ----------------------------------------------------

        x = x + self.ffn(
            self.norm3(x)
        )

        return x


# ============================================================
# 10. PRE-LN DECODER
# ============================================================

class TransformerDecoderPreLN(
    nn.Module
):

    def __init__(
        self,
        d_model=768,
        n_heads=8,
        d_ff=3072,
        n_layers=6,
        max_len=256,
        dropout=0.1
    ):
        super().__init__()

        self.layers = nn.ModuleList([
            TransformerDecoderLayerPreLN(
                d_model=d_model,
                n_heads=n_heads,
                d_ff=d_ff,
                dropout=dropout
            )
            for _ in range(n_layers)
        ])

        # Kept for diagnostics/check compatibility.
        self.register_buffer(
            "causal_mask",
            torch.triu(
                torch.full(
                    (max_len, max_len),
                    float("-inf")
                ),
                diagonal=1
            ),
            persistent=False
        )

    def forward(
        self,
        x,
        memory
    ):

        for layer in self.layers:

            x = layer(
                x,
                memory
            )

        return x


# ============================================================
# 11. COMPLETE PRE-LN MODEL
# ============================================================

class TransformerModelPreLN(
    nn.Module
):

    def __init__(
        self,
        vocab_size,
        max_src_len,
        max_tgt_len,
        d_model=768,
        n_heads=8,
        d_ff=3072,
        n_layers=6,
        dropout=0.1
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len

        # Shared embedding.
        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        # Fixed source positions.
        self.register_buffer(
            "src_pos_encoding",
            make_sinusoidal_table(
                max_src_len,
                d_model
            ),
            persistent=False
        )

        # Fixed target positions.
        self.register_buffer(
            "tgt_pos_encoding",
            make_sinusoidal_table(
                max_tgt_len,
                d_model
            ),
            persistent=False
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.encoder = TransformerEncoderPreLN(
            d_model=d_model,
            n_heads=n_heads,
            d_ff=d_ff,
            n_layers=n_layers,
            dropout=dropout
        )

        self.decoder = TransformerDecoderPreLN(
            d_model=d_model,
            n_heads=n_heads,
            d_ff=d_ff,
            n_layers=n_layers,
            max_len=max_tgt_len,
            dropout=dropout
        )

        # Tied output head.
        self.output_projection = nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )

        self.output_projection.weight = (
            self.token_embedding.weight
        )

        self._init_weights()

    def _init_weights(self):

        nn.init.normal_(
            self.token_embedding.weight,
            mean=0.0,
            std=0.02
        )

        for module in self.modules():

            if isinstance(
                module,
                nn.Linear
            ):

                if (
                    module.weight
                    is not self.token_embedding.weight
                ):

                    nn.init.xavier_uniform_(
                        module.weight
                    )

                if module.bias is not None:
                    nn.init.zeros_(
                        module.bias
                    )

            elif isinstance(
                module,
                nn.LayerNorm
            ):

                nn.init.ones_(
                    module.weight
                )

                nn.init.zeros_(
                    module.bias
                )

    def encode(self, src):

        _, T = src.shape

        assert T <= self.max_src_len

        x = self.token_embedding(src)

        x = x * math.sqrt(
            self.d_model
        )

        x = (
            x
            + self.src_pos_encoding[
                :T
            ].unsqueeze(0)
        )

        x = self.dropout(x)

        return self.encoder(x)

    def decode(
        self,
        tgt_in,
        memory
    ):

        _, T = tgt_in.shape

        assert T <= self.max_tgt_len

        x = self.token_embedding(
            tgt_in
        )

        x = x * math.sqrt(
            self.d_model
        )

        x = (
            x
            + self.tgt_pos_encoding[
                :T
            ].unsqueeze(0)
        )

        x = self.dropout(x)

        x = self.decoder(
            x,
            memory
        )

        return self.output_projection(x)

    def forward(
        self,
        src,
        tgt_in
    ):

        memory = self.encode(src)

        return self.decode(
            tgt_in,
            memory
        )


# ============================================================
# 12. BUILD MODEL
# ============================================================

def build_model(
    vocab_size,
    src_len,
    tgt_len
):

    assert src_len + tgt_len == 1024

    return TransformerModelPreLN(
        vocab_size=vocab_size,
        max_src_len=src_len,
        max_tgt_len=tgt_len,
        d_model=768,
        n_heads=8,
        d_ff=3072,
        n_layers=6,
        dropout=0.1
    )


# ============================================================
# 13. PARAMETER CHECK
# ============================================================

model = build_model(
    V,
    S,
    TT
).to(DEV)

n_params = sum(
    p.numel()
    for p in model.parameters()
)

print(
    f"\nPARAMETERS: "
    f"{n_params:,} "
    f"({n_params / 1e6:.2f}M)"
)

assert n_params == 129_957_888

print(
    "PASS parameter count = 129,957,888"
)

assert (
    50_000_000
    <= n_params
    <= 300_000_000
)

print(
    "PASS model is inside 50M-300M range"
)


# ============================================================
# 14. DATA
# ============================================================

def mm(name):

    path = f"{TOKD}/{name}.bin"

    assert os.path.exists(path), path

    return np.memmap(
        path,
        dtype=np.uint16,
        mode="r"
    )


data = {
    "acts": mm("acts_train"),
    "sc": mm("sc_train"),
    "tri": mm("tribunals_train_f"),
}

valbins = {
    "acts_val": mm("acts_val"),
    "sc_val": mm("sc_val"),
    "tribunals_val_f": mm("tribunals_val_f"),
}

keys = list(
    CFG["MIX"].keys()
)

probs = np.array(
    list(CFG["MIX"].values()),
    dtype=np.float64
)

probs = probs / probs.sum()


# ============================================================
# 15. WINDOW SPLIT
# ============================================================

def split_window(w):

    w = w[:, :BLOCK]

    src = w[:, :S]

    target = w[:, S:]

    bos = torch.full(
        (
            w.size(0),
            1
        ),
        BOS,
        dtype=w.dtype,
        device=w.device
    )

    tgt_in = torch.cat(
        [
            bos,
            target[:, :-1]
        ],
        dim=1
    )

    return src, tgt_in, target


# ============================================================
# 16. TRAINING BATCH
# ============================================================

def get_batch():

    windows = []

    for _ in range(MB):

        key = np.random.choice(
            keys,
            p=probs
        )

        d = data[key]

        start = np.random.randint(
            0,
            len(d) - BLOCK - 1
        )

        windows.append(
            np.asarray(
                d[
                    start:
                    start + BLOCK
                ],
                dtype=np.int64
            )
        )

    w = torch.from_numpy(
        np.stack(windows)
    ).to(DEV)

    return split_window(w)


# ============================================================
# 17. VALIDATION WINDOWS
# ============================================================

rs = np.random.RandomState(123)

val_windows = {}

for name, d in valbins.items():

    starts = rs.randint(
        0,
        len(d) - BLOCK - 1,
        200
    )

    val_windows[name] = np.stack([
        np.asarray(
            d[
                i:
                i + BLOCK
            ],
            dtype=np.int64
        )

        for i in starts
    ])


# ============================================================
# 18. PREFLIGHT
# ============================================================

@torch.no_grad()
def preflight():

    model.eval()

    src = torch.randint(
        0,
        V,
        (2, S),
        device=DEV
    )

    tin = torch.randint(
        0,
        V,
        (2, TT),
        device=DEV
    )

    with torch.autocast(
        "cuda",
        dtype=torch.float16
    ):

        out = model(
            src,
            tin
        ).float()

    assert out.shape == (
        2,
        TT,
        V
    )

    assert torch.isfinite(
        out
    ).all()

    print(
        "[preflight] shape OK:",
        tuple(out.shape)
    )

    # --------------------------------------------------------
    # Decoder causality
    # --------------------------------------------------------

    t = TT // 2

    tin2 = tin.clone()

    tin2[:, t:] = torch.randint(
        0,
        V,
        tin2[:, t:].shape,
        device=DEV
    )

    with torch.autocast(
        "cuda",
        dtype=torch.float16
    ):

        out2 = model(
            src,
            tin2
        ).float()

    leak = (
        out[:, :t]
        - out2[:, :t]
    ).abs().max().item()

    print(
        f"[preflight] decoder causality: "
        f"{leak:.2e}"
    )

    assert leak < 1e-3

    # --------------------------------------------------------
    # Encoder dependence
    # --------------------------------------------------------

    src2 = torch.randint(
        0,
        V,
        (2, S),
        device=DEV
    )

    with torch.autocast(
        "cuda",
        dtype=torch.float16
    ):

        out3 = model(
            src2,
            tin
        ).float()

    dep = (
        out
        - out3
    ).abs().max().item()

    print(
        f"[preflight] encoder dependence: "
        f"{dep:.2e}"
    )

    assert dep > 0

    # --------------------------------------------------------
    # Initial loss
    # --------------------------------------------------------

    src_b, tin_b, tout_b = get_batch()

    with torch.autocast(
        "cuda",
        dtype=torch.float16
    ):

        lg = model(
            src_b,
            tin_b
        )

    l0 = F.cross_entropy(
        lg.float().reshape(-1, V),
        tout_b.reshape(-1)
    ).item()

    print(
        f"[preflight] initial loss: "
        f"{l0:.4f} "
        f"(uniform={math.log(V):.4f})"
    )

    model.train()


preflight()


# ============================================================
# 19. OPTIMIZER
# ============================================================

try:

    opt = torch.optim.AdamW(
        model.parameters(),
        lr=CFG["LR"],
        weight_decay=CFG["WD"],
        fused=True
    )

    print(
        "Optimizer: fused AdamW"
    )

except Exception as e:

    print(
        "WARNING: fused AdamW unavailable"
    )

    print(
        repr(e)
    )

    opt = torch.optim.AdamW(
        model.parameters(),
        lr=CFG["LR"],
        weight_decay=CFG["WD"]
    )


# ============================================================
# 20. AMP
# ============================================================

scaler = torch.amp.GradScaler(
    "cuda"
)


# ============================================================
# 21. LR SCHEDULE
# ============================================================

def lr_at(step):

    if step < CFG["WARMUP"]:

        return (
            CFG["LR"]
            * (step + 1)
            / CFG["WARMUP"]
        )

    p = (
        step
        - CFG["WARMUP"]
    ) / max(
        1,
        CFG["STEPS"]
        - CFG["WARMUP"]
    )

    p = min(
        max(
            p,
            0.0
        ),
        1.0
    )

    return CFG["LR"] * (
        CFG["MIN_LR_FRAC"]
        + (
            1
            - CFG["MIN_LR_FRAC"]
        )
        * 0.5
        * (
            1
            + math.cos(
                math.pi * p
            )
        )
    )


# ============================================================
# 22. TRAIN STEP
# ============================================================

def train_step(lr):

    for group in opt.param_groups:

        group["lr"] = lr

    total = 0.0

    for _ in range(GA):

        src, tin, tout = get_batch()

        with torch.autocast(
            "cuda",
            dtype=torch.float16
        ):

            logits = model(
                src,
                tin
            )

            loss = (
                F.cross_entropy(
                    logits.float().reshape(
                        -1,
                        V
                    ),
                    tout.reshape(-1)
                )
                / GA
            )

        scaler.scale(
            loss
        ).backward()

        total += loss.item()

        del (
            src,
            tin,
            tout,
            logits,
            loss
        )

    scaler.unscale_(opt)

    grad_norm = (
        nn.utils.clip_grad_norm_(
            model.parameters(),
            CFG["CLIP"]
        )
        .item()
    )

    scaler.step(
        opt
    )

    scaler.update()

    opt.zero_grad(
        set_to_none=True
    )

    return total, float(
        grad_norm
    )


# ============================================================
# 23. VALIDATION
# ============================================================

@torch.no_grad()
def val_loss(bs=2):

    model.eval()

    results = {}

    for name, windows in val_windows.items():

        total = 0.0
        count = 0

        for i in range(
            0,
            len(windows),
            bs
        ):

            b = torch.from_numpy(
                windows[
                    i:
                    i + bs
                ]
            ).to(DEV)

            src, tin, tout = (
                split_window(b)
            )

            with torch.autocast(
                "cuda",
                dtype=torch.float16
            ):

                logits = model(
                    src,
                    tin
                )

            total += F.cross_entropy(
                logits.float().reshape(
                    -1,
                    V
                ),
                tout.reshape(-1),
                reduction="sum"
            ).item()

            count += tout.numel()

        results[name] = (
            total / count
        )

    model.train()

    return results


# ============================================================
# 24. CHECKPOINT
# ============================================================

def save_ckpt(
    path,
    step,
    best
):

    torch.save(
        {
            "model":
                model.state_dict(),

            "opt":
                opt.state_dict(),

            "scaler":
                scaler.state_dict(),

            "step":
                step,

            "best":
                best,

            "cfg":
                CFG,

            "n_params":
                n_params,
        },
        path
    )


# ============================================================
# 25. FRESH TRAINING ONLY
#
# There is intentionally NO checkpoint loading here.
# ============================================================

print("\n" + "=" * 70)
print("STARTING COMPLETELY FRESH PRE-LN TRAINING")
print("=" * 70)

print("RUN:", RUN)
print("No old checkpoint will be loaded.")
print(
    f"Parameters: "
    f"{n_params:,} "
    f"({n_params/1e6:.2f}M)"
)

print(
    f"Steps={CFG['STEPS']:,} "
    f"LR={CFG['LR']} "
    f"warmup={CFG['WARMUP']}"
)

print(
    f"Micro-batch={MB} "
    f"Grad-accum={GA} "
    f"Effective={MB*GA}"
)

print("=" * 70)


model.train()

start_time = time.time()
last_time = start_time
last_step = 0

history = []

best = float("inf")

for step in range(
    CFG["STEPS"]
):

    lr = lr_at(step)

    loss, grad_norm = (
        train_step(lr)
    )

    history.append(loss)

    elapsed = (
        time.time()
        - start_time
    )

    timeout = (
        elapsed
        >= TIME_LIMIT_H * 3600
        if "TIME_LIMIT_H" in globals()
        else elapsed >= 8.25 * 3600
    )

    should_eval = (
        (step + 1)
        % CFG["EVAL_EVERY"]
        == 0
        or
        (step + 1)
        == CFG["STEPS"]
        or
        timeout
    )

    if should_eval:

        val = val_loss(
            bs=2
        )

        mean_val = float(
            np.mean(
                list(
                    val.values()
                )
            )
        )

        now = time.time()

        interval_time = max(
            now - last_time,
            1e-9
        )

        interval_steps = (
            step + 1
            - last_step
        )

        tok_s = (
            interval_steps
            * MB
            * GA
            * BLOCK
            / interval_time
        )

        seen = (
            (step + 1)
            * MB
            * GA
            * BLOCK
        )

        train_mean = float(
            np.mean(
                history[-50:]
            )
        )

        peak_gb = (
            torch.cuda
            .max_memory_allocated()
            / 1e9
        )

        print(
            f"\nSTEP "
            f"{step+1:,}/"
            f"{CFG['STEPS']:,}"
            f" | "
            f"{seen/1e6:,.1f}M tok"
            f" | train "
            f"{train_mean:.4f}"
            f" | "
            + " ".join(
                f"{k}={v:.4f}"
                for k, v in val.items()
            )
            + f" | mean "
            f"{mean_val:.4f}"
            + f" | "
            f"{tok_s:,.0f} tok/s"
            + f" | "
            f"{peak_gb:.2f} GB"
            + f" | lr {lr:.2e}"
            + f" | grad {grad_norm:.3f}"
        )

        record = {
            "step":
                step + 1,

            "tokens_consumed":
                int(seen),

            "target_tokens":
                int(
                    (step + 1)
                    * MB
                    * GA
                    * TT
                ),

            "train_loss":
                train_mean,

            "val_mean":
                mean_val,

            "lr":
                float(lr),

            "grad_norm":
                float(
                    grad_norm
                ),

            "tok_per_sec":
                float(tok_s),

            **val,
        }

        with open(
            f"{RUN}/log.jsonl",
            "a"
        ) as f:

            f.write(
                json.dumps(record)
                + "\n"
            )

        if mean_val < best:

            best = mean_val

            save_ckpt(
                f"{RUN}/best.pt",
                step + 1,
                best
            )

            print(
                f"NEW BEST: "
                f"{best:.4f}"
            )

        save_ckpt(
            f"{RUN}/latest.pt",
            step + 1,
            best
        )

        last_time = now
        last_step = step + 1

    if timeout:

        print(
            "\nTIME GUARD:"
            " checkpoint saved."
        )

        break


# ============================================================
# 26. FINAL STATUS
# ============================================================

status = (
    "DONE"
    if step + 1 >= CFG["STEPS"]
    else "PAUSED"
)

print("\n" + "=" * 70)
print("FINAL STATUS:", status)
print("RUN:", RUN)
print("BEST VAL:", best)
print("=" * 70)

for filename in [
    "best.pt",
    "latest.pt",
    "log.jsonl"
]:

    path = os.path.join(
        RUN,
        filename
    )

    print(
        f"{filename:10s}",
        "OK"
        if os.path.exists(path)
        else "MISSING"
    )

print(
    "\nIMPORTANT:"
    " this run is a fresh Pre-LN model;"
    " the previous run_arch2_encdec checkpoint"
    " was not loaded."
)

CUDA: Tesla T4
Config: SRC=768 TGT=256 MB=2 GA=8 effective=16
RUN: /kaggle/working/run_arch2_preln_efficient
FRESH RUN: no previous checkpoint will be loaded.
BOS token: 1

PARAMETERS: 129,957,888 (129.96M)
PASS parameter count = 129,957,888
PASS model is inside 50M-300M range
[preflight] shape OK: (2, 256, 40000)
[preflight] decoder causality: 0.00e+00
[preflight] encoder dependence: 1.54e+00
[preflight] initial loss: 15.2055 (uniform=10.5966)
Optimizer: fused AdamW

STARTING COMPLETELY FRESH PRE-LN TRAINING
RUN: /kaggle/working/run_arch2_preln_efficient
No old checkpoint will be loaded.
Parameters: 129,957,888 (129.96M)
Steps=10,000 LR=0.0003 warmup=500
Micro-batch=2 Grad-accum=8 Effective=16

STEP 500/10,000 | 8.2M tok | train 5.9119 | acts_val=5.5827 sc_val=5.7676 tribunals_val_f=5.9728 | mean 5.7744 | 20,855 tok/s | 3.18 GB | lr 3.00e-04 | grad 7.410
NEW BEST: 5.7744

STEP 1,000/10,000 | 16.4M tok | train 4.8445 | acts_val=4.6303 sc_val=4.8313 tribunals_val_f=4.8592 | mean 4.7736 

In [3]:
sdfs

NameError: name 'sdfs' is not defined

In [ ]:
avfdd